# Manual Trial-and-Error

In the current notebook we will explore some concepts through manual history-matching (or trial-and-error). Many of these concepts are explained in greater detail in *Applied Groundwater Modeling (2nd edition)* by [Anderson et al. (2015)](https://www.sciencedirect.com/book/9780120581030/applied-groundwater-modeling). Here, we will "manually" adjust model parameter values, run the model, and then compare model outputs to measured values. This process is repeated until the modeller is satisfied with the fit between measured and simulated values.

Before we go any further, allow us to highlight that trial-and-error history matching is rarely, if ever, sufficient in a decision-support modelling context. It can be a useful **first-step** for history-matching, mostly because it provides a modeller with insight about the site and a model's response to parameter changes. In this way, it helps to develop a modeller's "hydrosense" - the intuitive understanding of how a model behaves. It can also provide a quick form of quality control on both the model setup as well as the reasonableness of the conceptual model.

These benefits notwithstanding, trial-and-error history matching is cumbersome and highly subjective. It is inapplicable in highly-parameterized contexts (we will see why this is important in other tutorials). Comprehensive testing and identification of all insensitive and correlated parameters is not feasible, and it cannot ensure that the best quantifiable fit has been achieved. 

When undertaking decision-support modelling in practice, more rigorous, automated trial-and-error methodologies are employed. Such methods are the topics of subsequent tutorials.

**Key Point:** 
>Trial-and-error history-matching may provide some "soft" benefit to the modeller. It does not replace formal statistical parameter estimation.

**References:**

>Anderson, Mary P., William W. Woessner, and Randall J. Hunt. 2015. *Applied Groundwater Modeling*. Applied Groundwater Modeling. 2nd ed. Elsevier. doi:10.1016/B978-0-08-091638-5.00001-8.




### Admin

In the tutorial folder there is a file named `freyberg_trial_and_error.py`. It contains a series of functions that automate changing parameters, running the model and plotting outcomes. These make use of `flopy` and other libraries. You do not need to be familiar with these functions, only to follow along with what they are doing.

In [ ]:
import sys
import os
import warnings
warnings.filterwarnings("ignore")
warnings.filterwarnings("ignore", category=DeprecationWarning) 

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt;
plt.rcParams.update({'font.size': 12})

import shutil

import freyberg_trial_and_error as te
#prepares some files for manunal trial and error
te.get_model()

# Trial and Error

The `te.update_par()` function loads the modified Freyberg model (see the `intro_freyberg_model` notebook), updates parameters, runs the model and then plots simulated values against measured values.

The function automates updates to:

- hydraulic conductivity (k)
- recharge

You can assign a homogeneous values of k through the `k1` argument. You can adjust recharge by passing a value to the `rch_factor` argument. Recharge in the model is multiplied by this factor. 

Scatter plots of measured *vs* simulated values are displayed for each observation type, along with the *objective function* - usually referred to as "phi". Phi is the weighted sum-of-squared residuals:

$\Phi = \sum_{i=1}^{n} \left[ w_i \left( y_i - y'_i \right) \right]^2$

where $y_i$ is the $i$-th measured value, $y'_i$ is the corresponding simulated value and $w_i$ is the weight assigned to that observation. This is the *same* measure of model-to-measurement fit that PESTPP-GLM and PESTPP-IES minimize in the tutorials that follow - so the number you are trying to make small here is the number those tools will be trying to make small later on. 

Weights do two jobs: they put observations of different types (heads in metres, flows in cubic metres per day) onto a common footing, and they let us express how much we trust - and how much we care about - each measurement. An observation with a weight of zero does not contribute to phi at all; it plays no role in history-matching.

Initially only head observations are weighted; the `sfr_weight` argument (the weight assigned to the river gage observations) defaults to zero. We will come back to that shortly.

For now we are looking *only* at the fit with measured data - just as we would be in the real world, where the future is not available for inspection. We will get to the forecasts at the end of the notebook.

For example:

In [ ]:
te.update_par(k1=1,     # K in layer 1
              rch_factor=0.35) # recharge is multiplied by rch_factor

### Do It Yourself

Experiment with changing values for `k1` and `rch_factor`. See if you can achieve a good fit between measured and simulated values of head (e.g. minimize phi). 

Keep track of the parameter values that give you the lowest phi - we will want them later. Are both parameters equally effective at reducing phi? 

> Note: you have now seen MODFLOW's run output once, which is how you check that the model actually ran. From here on we pass `silent=True` to keep the notebook readable - the model still runs (and a failed run will still raise an error), we just don't echo the report. Drop the argument any time you want to see it again.

In [ ]:
# change the parameter values until you are happy with the fit
te.update_par(k1=8, rch_factor=1, silent=True)

### Non-Uniqueness and Correlated Parameters

So you may have found that values of `k1`=4, and `rch_factor`=1.1 provide a decent fit.

...but hold on! What's this? If we assign `k1`=10,  `rch_factor`=2, phi is very similar. Oh dear.

So which one is correct? Nothing in the measured data can tell us - both parameter sets reproduce it about equally well. This is *non-uniqueness*, and it is a direct consequence of k and recharge being correlated: increasing recharge and increasing k in the right proportions leaves simulated heads more-or-less unchanged. Hold that thought; we will see what it costs us when we get to the forecasts.

In [ ]:
te.update_par(k1=4, rch_factor=1.1, silent=True)

In [ ]:
te.update_par(k1=10,  rch_factor=2.0, silent=True)


One option is to use multiple types of observation data. Using heads & flows as calibration targets helps to constrain parameters which are informed by different sources of information. The same applies for secondary observations; e.g. vertical head differences for vertical connectivity and time-differences for storage or transport parameters.

To bring the stream gage data into history-matching, we have to give it a non-zero weight. But *what* weight? There is no equation that will hand you the answer. Flows here are numerically ~100 times larger than heads (cubic metres per day *vs* metres), so a weight of 1.0 for both would let the flux observations swamp phi entirely and the heads would effectively stop mattering.

**Our suggestion: use `sfr_weight=0.003`.** A common way to arrive at a number like this is to set the weight to the inverse of the measurement error: if we reckon the gage is good to about 10% of mean flow (mean flow is ~2500 $\frac{m^3}{d}$), that gives a weight of about 1/250 = 0.004. We then nudged it down to 0.003 - not because the data said so, but because it worked better here: it keeps the head and flux contributions to phi within the same order of magnitude, rather than letting one drown out the other (a weight of 0.05 would have the flux contributing ~1000 times more than the heads). This is the value we use for the streamflow observations throughout the rest of the part1 tutorials, so the fits you achieve here by hand can be compared to the fits PEST++ achieves later. (In code it is `te.SFR_WEIGHT` here, and `hbd.SFR_WEIGHT` in the notebooks that follow, if you would rather change it in one place.)

Be clear-eyed about what just happened though: we *chose* that number. "About 10% of mean flow" was a judgement, not a measurement. Try `sfr_weight=0.05`, or `sfr_weight=0.0001`, and watch phi - and the parameter values that minimize it - change. Weighting is a statement about how much each measurement matters, and it is one of the most consequential and least defensible decisions a modeller makes. We will return to it in much more detail in part2.

See if accounting for the fit with stream gage data helps:

In [ ]:
te.update_par(k1=4, rch_factor=1.1, sfr_weight=te.SFR_WEIGHT, silent=True)

## And Now, the Forecasts

Everything so far has been about the *past* - how well the model reproduces measurements we already have. But nobody builds a model to reproduce the past. We build it to say something about the future.

Our model is used to make forecasts at the end of the simulation period. Three of them are shown here (they are amongst the forecasts we will carry through the rest of the part1 tutorials):

- `HEADWATER`: groundwater/surface-water exchange flux in the headwater reaches of the river
- `TAILWATER`: the same, but in the tailwater reaches
- `TRGW-0-9-1`: groundwater level at a well in the north-west of the domain

Because this is a synthetic model, *we* generated reality - so we know what each of these forecasts should be. In the real world we obviously do not. Passing `plot_forecast=True` to `te.update_par()` adds a bar chart of the **forecast error** (simulated minus truth) for each forecast. A perfect model would have bars of zero height.

Let's take that decent fit from before and see how it did:

In [ ]:
te.update_par(k1=4, rch_factor=1.1, sfr_weight=te.SFR_WEIGHT, plot_forecast=True, silent=True)

Now remember that other parameter set - the one that fit the *head* data just as well. If the two are indistinguishable in terms of the heads, are they also indistinguishable in terms of the forecasts?

In [ ]:
te.update_par(k1=10, rch_factor=2.0, sfr_weight=te.SFR_WEIGHT, plot_forecast=True, silent=True)

Not remotely. Two parameter sets that the head observations could not tell apart give wildly different forecasts - and they are wrong in *opposite directions*. A modeller history-matching to heads alone could have picked either one, and would have had no way of knowing which.

Notice also what the gage observations did here: with the flux data weighted, these two parameter sets are no longer equivalent at all - the second one has a composite phi roughly 35 times worse. This is the payoff for using multiple types of observation data. The flux data carries information about the forecasts that the heads alone simply do not.

### Structural Error

But look again at the forecast error bars for `k1`=4, `rch_factor`=1.1. That was our *best* fit, and the forecasts are still wrong. So try it the other way around: see if you can find a parameter combination that results in a "correct" forecast (bars near zero). What does the corresponding phi look like? Is it the parameter set you would have chosen on the basis of the fit alone?

In [ ]:
# change the parameter values and see what it does to the forecast error
te.update_par(k1=4, rch_factor=1.1, sfr_weight=te.SFR_WEIGHT, plot_forecast=True, silent=True)

So...is history-matching a lie? It seems to make our model worse at making a prediction! 

Well, in this case it does. This is because our model is *oversimplified*. We are introducing *structural error* by using a single parameter value for each model layer, ignoring the potential for parameter heterogeneity. We touched on this in the "intro to regression" notebook and we will revisit it again in other tutorials.

## Final Remarks

Hopefully after this gruelling exercise you have learned the value (and lack-thereof) of manual trial-and-error history-matching. We have seen that:

1. We can gain some insight into the model (e.g. confirming the first-principles knowledge that K and R are correlated), but more complex insights can be masked by structural error and an inability to explore all parameters as complexity increases. After some initial effort to make sure a model can even run with initial parameters, trial-and-error becomes inadequate.
2. Decisions on which parameters to change, in what order and by how much is subjective and will be subject to each modeller's biases. Thus trial-and-error history matching does not provide a transparent and reproducible process. Nor does it provide a quantifiable "best fit" for a given model.
3. Here we adjusted 2 parameters. As we saw, this *oversimplified* parameterisation introduced structural error, biasing our prediction. History-matching such a model can actually harm its usefulness to support a decision. Increasing the number of parameters reduces this propensity for bias. But handling more parameters manually quickly becomes an impossible task.
4. We saw that certain combinations of parameters can provide similar, or even equal, fits with measured data. Manually identifying parameter correlation is challenging, to say the least. And, once identified, a modeller is forced to (subjectively) choose which parameters to fix, and which to adjust.
5. Perhaps most importantly, once again we saw that a good fit with measured data does not equate to a correct prediction. To be useful in a decision-support context, prediction uncertainty must be characterized.
